In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CRYPTOGRAPHY_OPENSSL_NO_LEGACY'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import datasets
from datasets import load_dataset, Dataset, DatasetDict, Dataset
from tqdm.auto import tqdm
import json

datasets.disable_caching()

/mnt/nas/home/genggui/miniconda3/envs/megatron-next/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from megatron.energon import get_train_dataset, WorkerConfig, get_loader, DefaultTaskEncoder, Cooker, basic_sample_keys
from megatron.energon import DefaultTaskEncoder, TextSample, Sample
import gzip
import pickle
import webdataset as wds
import json
import dataclasses
import torch

from tqdm.auto import tqdm
from typing import List

from qwen3_base_energon_helpers import MyTaskEncoder, print_error_handler

In [3]:
tokenizer_path="/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_122b_a10b_next_gemini_bf16/tokenizer_v19_gemini"

In [4]:
worker_config = WorkerConfig(
    rank=0,
    world_size=1,
    seed_offset=123456789,
    num_workers=1,
)

train_ds = get_train_dataset(
    # '/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_122b_a10b_next_gemini_bf16/dataset.yaml',
    "/mnt/nas/home/genggui/dataset/nlp/new_think_dd/sft_used_data/tools/cn/MSAgent-Bench-en-think",
    split_part="train",
    task_encoder=MyTaskEncoder(
        tokenizer_path=tokenizer_path,
        seq_length=66536,
        sensitive_words_path=None,
    ),
    batch_size=1,
    packing_buffer_size=8192,
    shuffle_buffer_size=None,
    max_samples_per_sequence=None,
    worker_config=worker_config,
)

train_dataloader = get_loader(train_ds, worker_config=worker_config, watchdog_timeout_seconds=240)

rank=0, worker=0: sample_range=[0, 14199] in 1 slices, sum(count)=14199: indexes=[0, 14199] slices=[train-chunk-0.tar[0(start), 14199(end)]]


/mnt/nas/home/genggui/miniconda3/envs/megatron-next/lib/python3.12/site-packages/megatron/energon/loader.py:108: FutureWarning: Passing a worker_config to get_loader() is deprecated and will have no effect.
  warn_deprecated(


In [ ]:
dd = iter(train_dataloader)

Filling reading buffer:   0%|          | 11/8192 [00:00<01:16, 106.62it/s]

load tool_calls arguments error, message_tool_calls: [{'function': {'arguments': [{'name': 'movie_name', 'value': '肖申克的救赎'}], 'name': 'movie_info'}, 'id': None, 'index': None, 'type': 'function'}], error: tool_call arguments type error: <class 'list'>


Filling reading buffer:  26%|██▌       | 2135/8192 [00:06<00:21, 283.61it/s]

load tool_calls arguments error, message_tool_calls: [{'function': {'arguments': [{'name': 'type', 'value': '喜剧'}, {'name': 'page', 'value': 1}], 'name': 'movie'}, 'id': None, 'index': None, 'type': 'function'}], error: tool_call arguments type error: <class 'list'>


Filling reading buffer:  30%|███       | 2493/8192 [00:07<00:19, 291.92it/s]

...

Filling reading buffer:  59%|█████▉    | 4842/8192 [00:15<00:10, 328.95it/s]

load tool_calls arguments error, message_tool_calls: [{'function': {'arguments': [{'company': '华为', 'date': '2021-09-01'}], 'name': 'stocks'}, 'id': None, 'index': None, 'type': 'function'}], error: tool_call arguments type error: <class 'list'>


Filling reading buffer:  62%|██████▏   | 5109/8192 [00:16<00:10, 291.13it/s]

load tool_calls arguments error, message_tool_calls: [{'function': {'arguments': [{'name': 'keyword', 'value': '鱼香肉丝'}, {'name': 'difficulty', 'value': '初级'}], 'name': 'cooking_recipes'}, 'id': None, 'index': None, 'type': 'function'}], error: tool_call arguments type error: <class 'list'>


Filling reading buffer: 100%|██████████| 8192/8192 [00:26<00:00, 311.93it/s]


In [ ]:
for _ in tqdm(range(2048)):
    item = next(dd)

In [ ]:
import json

count_map = {}

with open("/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_122b_a10b_next_gemini_bf16/qua/pulse_v19_122b_a10b_gemini_train.jsonl", "w", encoding="utf8") as f:
    for _ in tqdm(range(2048)):
        item = next(dd)
        
        kk = item['__key__'][0].split("/")[-1].split("-train-")[0]
        if kk not in count_map:
            count_map[kk] = 0
        
        count_map[kk] += 1
              
        item = {
            "input_ids": item['input_ids'].tolist()[0],
            "labels": item['labels'].tolist()[0],
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        

            
        

In [ ]:
count_list = sorted(count_map.items(), key=lambda x:-x[1])
[(k,v/2048)for k,v in count_list]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [ ]:
tokenizer("<|im_start|>assistant\n<ggthink>\n\n</ggthink>\n\n").input_ids 

In [ ]:
tokenizer("<|im_start|>assistant\n<ggthink>\n").input_ids

In [ ]:
tokenizer('<|im_end|>').input_ids

In [ ]:
for _ in tqdm(range(2048)):
    item = next(dd)

In [ ]:
item = next(dd)
# print(item['__key__'][0].split("/")[-1].split("-train-")[0])
print(tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100]))

In [ ]:
print(tokenizer.decode([item for item in item['labels'].tolist()[0] if item != -100]))